# 12 · Nested Cross-Validation

## What this notebook covers
Hyperparameter tuning and performance estimation on the same data introduces optimistic bias. Nested CV solves this by separating the two concerns with two loops: the inner loop tunes, the outer loop evaluates.

## Why the bias exists
If you do `GridSearchCV` then report the best model's score on the same CV folds, you've used the test data to select hyperparameters. The reported score is optimistic because the model was chosen specifically because it happened to do well on those folds.

## The nested CV structure
```
Outer loop (k_outer folds):
  for each outer fold:
    Inner loop (k_inner folds) on outer train set:
      tune hyperparameters via CV
    Refit best params on full outer train set
    Evaluate on outer test fold  ← unbiased estimate
```

The outer scores give an unbiased estimate of generalisation performance. The final deployed model is typically retrained on all data using the hyperparameters most commonly selected by the inner loop.

## When to use it
Use nested CV whenever you are tuning and reporting performance. The overhead is k_outer × k_inner × n_candidates model fits — for large models, use fewer outer folds (3-5) and a coarse inner search.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    KFold, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=0)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

In [ ]:
# ❌ Naive: tune + evaluate on same CV folds → optimistic bias
param_grid = {"n_estimators": [50, 100, 200], "max_depth": [2, 3, 5], "learning_rate": [0.05, 0.1]}
clf_base = GradientBoostingClassifier(random_state=0)
gs = GridSearchCV(clf_base, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
naive_scores = cross_val_score(gs, X, y, cv=5, scoring="roc_auc")
print(f"Naive (biased)  AUC: {naive_scores.mean():.4f} ± {naive_scores.std():.4f}")

In [ ]:
# ✅ Nested CV: inner loop tunes, outer loop evaluates
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

outer_scores = []
best_params_list = []

for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y)):
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    gs_inner = GridSearchCV(
        GradientBoostingClassifier(random_state=0),
        param_grid, cv=inner_cv, scoring="roc_auc", n_jobs=-1
    )
    gs_inner.fit(X_tr, y_tr)
    best_params_list.append(gs_inner.best_params_)

    score = roc_auc_score(y_te, gs_inner.best_estimator_.predict_proba(X_te)[:, 1])
    outer_scores.append(score)
    print(f"Fold {fold+1}: AUC={score:.4f}  best_params={gs_inner.best_params_}")

outer_scores = np.array(outer_scores)
print(f"\nNested CV (unbiased) AUC: {outer_scores.mean():.4f} ± {outer_scores.std():.4f}")
print(f"Optimism (naive - nested): {naive_scores.mean() - outer_scores.mean():.4f}")

In [ ]:
# Most commonly selected hyperparameters → use these for final model
params_df = pd.DataFrame(best_params_list)
print("\nHyperparameter selection frequency across outer folds:")
print(params_df.apply(pd.Series.value_counts).fillna(0).astype(int).to_string())

# Final model: refit on all data with modal hyperparams
from scipy.stats import mode
final_params = {col: mode(params_df[col], keepdims=True).mode[0] for col in params_df.columns}
print(f"\nFinal params for production model: {final_params}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(range(1, 6), outer_scores, color="steelblue", alpha=0.8)
axes[0].axhline(outer_scores.mean(), color="red", linestyle="--", label=f"Mean={outer_scores.mean():.3f}")
axes[0].axhline(naive_scores.mean(), color="orange", linestyle=":", label=f"Naive={naive_scores.mean():.3f}")
axes[0].set_xlabel("Outer fold")
axes[0].set_ylabel("AUC")
axes[0].set_title("Nested CV: per-fold AUC")
axes[0].legend()
axes[0].set_ylim(0.7, 1.0)

# Variance of outer scores → confidence in the estimate
axes[1].hist(outer_scores, bins=5, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].axvline(outer_scores.mean(), color="red", lw=2, label=f"Mean {outer_scores.mean():.3f}")
axes[1].set_title("Outer fold score distribution")
axes[1].set_xlabel("AUC")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{base}/12_nested_cv.png", dpi=120)
plt.show()

## Summary
- Nested CV gives an unbiased performance estimate when hyperparameters are tuned
- The gap between naive and nested scores is the *optimism* — typically 0.005–0.03 AUC for small datasets
- Use 5×3 outer/inner folds as a default; reduce outer to 3 for expensive models
- The final model is retrained on all data using the most-selected hyperparameters from the inner loop